# EDA: Реєстр платників ПДВ України

**Опис:** первинний огляд та підготовка даних реєстру платників ПДВ (Open Data).

**Мета:** перевірити структуру, якість даних і підготувати базову візуалізацію для подальшого аналізу.

## Імпорти
Стандартні бібліотеки для аналізу та візуалізації.

In [ ]:
import sys
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path.cwd().parent))

from src.config import (
    DATA_FILE, CSV_SEP, CSV_ENCODING, CSV_ON_BAD_LINES,
    DATE_FORMAT, PLOT_STYLE, PANDAS_MAX_COLUMNS,
    ANALYSIS_START_YEAR, ECONOMIC_EVENT_YEAR,
)

from src.analysis import (
    analyze_seasonality,
    analyze_legal_forms,
    analyze_economic_impact
)

from src.visualizations import (
    plot_registrations_by_year,
    plot_seasonality,
    plot_legal_forms,
    plot_economic_impact
)

## Конфігурація
Налаштування відображення та стилю графіків.

In [ ]:
pd.set_option("display.max_columns", PANDAS_MAX_COLUMNS)
plt.style.use(PLOT_STYLE)

## Завантаження даних
Читаємо CSV з обробкою кодування та помилкових рядків.

In [ ]:
df = pd.read_csv(
    DATA_FILE,
    sep=CSV_SEP,
    encoding=CSV_ENCODING,
    on_bad_lines=CSV_ON_BAD_LINES,
)

df["dat_term"] = df["dat_term"].replace("null", pd.NA)
df["dat_reestr"] = pd.to_datetime(df["dat_reestr"], format=DATE_FORMAT, errors="coerce")

## Базовий огляд
Швидко перевіряємо перші рядки, типи полів та пропуски.

In [ ]:
print("Data head:")
df.head()

In [ ]:
print("Data info:")
df.info()

print("\nData missing values:")
df.isnull().sum()

## Візуалізація
Приклад: кількість реєстрацій за роками.

In [ ]:
registrations_by_year = (
    df.dropna(subset=["dat_reestr"])
    .assign(year=lambda x: x["dat_reestr"].dt.year)
    .groupby("year")
    .size()
    .sort_index()
 )

plot_registrations_by_year(registrations_by_year)

## Гіпотеза 1: Сезонність реєстрацій

**Припущення:** Реєстрація нових платників ПДВ має чітко виражену сезонність із піками на початку кожного кварталу (січень, квітень, липень, жовтень).

In [ ]:
results = analyze_seasonality(df)

print("Розподіл реєстрацій за місяцями:")
print(results["registrations_by_month"])
print("\nРозподіл реєстрацій за кварталами:")
print(results["registrations_by_quarter"])

plot_seasonality(results)

print("\n✓ Аналіз гіпотези:")
print(f"  Піковий місяць: {results['peak_month']} ({results['peak_count']:,} реєстрацій)")
print(f"  Найменш активний місяць: {results['low_month']} ({results['low_count']:,} реєстрацій)")
print("  Очікування: піки у місяцях 1, 4, 7, 10 (початок кварталів)")

## Гіпотеза 2: Зміна популярності організаційно-правових форм

**Припущення:** Популярність форми "Приватне підприємство" (ПП) серед нових платників ПДВ катастрофічно знизилася на користь "Товариств з обмеженою відповідальністю" (ТОВ) після 2010 року.

In [ ]:
results = analyze_legal_forms(df, event_year=2010)

print("Розподіл форм за роками та періодами:")
print("\nДо 2010 року:")
print(results["before_event"])
print(f"Всього: {results['before_event'].sum()}")

print(f"\nВід 2010 року:")
print(results["after_event"])
print(f"Всього: {results['after_event'].sum()}")

plot_legal_forms(results)

print("\nАналіз гіпотези:")
print(f"  ПП: {results['pp_before']:,} → {results['pp_after']:,} ({results['pp_change_pct']:+.1f}%)")
print(f"  ТОВ: {results['tov_before']:,} → {results['tov_after']:,} ({results['tov_change_pct']:+.1f}%)")
if results["pp_declined"] and results["tov_increased"]:
    print("Гіпотеза підтверджується!")
else:
    print("Гіпотеза не повністю підтверджується")

## Гіпотеза 3: Вплив економічних подій на реєстрацію

**Припущення:** Кількість нових реєстрацій платників ПДВ суттєво зросла після 2014 року порівняно з періодом 2010–2013 років.

In [ ]:
results = analyze_economic_impact(df)

print("Порівняння періодів:")
print(f"\n{results['before_event']['period']} роки:")
print(f"  Всього реєстрацій: {results['before_event']['total']:,}")
print(f"  Років у періоді: {results['before_event']['years']}")
print(f"  Середньо на рік: {results['before_event']['avg_per_year']:,.0f}")

print(f"\n{results['after_event']['period']}:")
print(f"  Всього реєстрацій: {results['after_event']['total']:,}")
print(f"  Років у періоді: {results['after_event']['years']}")
print(f"  Середньо на рік: {results['after_event']['avg_per_year']:,.0f}")

print(f"\nЗміна середньої кількості: {results['change_percent']:+.1f}%")

plot_economic_impact(results)

print("\n✓ Аналіз гіпотези:")
if results["hypothesis_confirmed"]:
    print(f"  Гіпотеза підтверджується: реєстрації зросли на {results['change_percent']:.1f}%")
else:
    print(f"  Гіпотеза не підтверджується: зміна {results['change_percent']:.1f}%")